In [ ]:
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key='XXXXX')


In [11]:
# List of ENTSO-E member country codes
COUNTRY_CODES = [ "HR" ]


# Define the time range
start = pd.Timestamp('20220101', tz='Europe/Brussels')
end = pd.Timestamp('20221231', tz='Europe/Brussels')
#start = pd.Timestamp('20211201', tz='Europe/Brussels')
#end = pd.Timestamp('20220301', tz='Europe/Brussels')
all_loads = {}

In [6]:
load_data = client.query_load(COUNTRY_CODES, start=start, end=end)

In [8]:
load_data1 = client.query_load(COUNTRY_CODES, start=start, end=end)

In [12]:
# Function to split date range into 3-month chunks
def split_date_range(start, end, chunk_months=3):
    current_start = start
    while current_start < end:
        current_end = min(current_start + pd.DateOffset(months=chunk_months), end)
        yield current_start, current_end
        current_start = current_end


# Fetch data in 3-month chunks per country
for country_code in COUNTRY_CODES:
    print(f"Fetching data for {country_code}...")

    # Process price data in 3-month chunks
    #price_chunks = []
    #for chunk_start, chunk_end in split_date_range(start, end, 3):
    #    try:
    #        price_data = client.query_day_ahead_prices(country_code, start=chunk_start, end=chunk_end)
#
    #        # Convert to DataFrame if needed
    #        if isinstance(price_data, list):
    #            price_data = pd.DataFrame(price_data)
#
    #        if not price_data.empty:
    #            price_chunks.append(price_data)
    #    except Exception as e:
    #        print(f"Failed to fetch price data for {country_code} from {chunk_start} to {chunk_end}: {e}")
#
    #if price_chunks:
    #    price_data = pd.concat(price_chunks)  # Merge chunks
    #    price_data = price_data.groupby(level=0).first()  # Remove duplicate timestamps
    #    all_prices[country_code] = price_data
    #else:
    #    # Ensure missing country is represented with NaN
    #    all_prices[country_code] = pd.Series(dtype=float, name=country_code)

    # Process load data in 3-month chunks
    load_chunks = []
    for chunk_start, chunk_end in split_date_range(start, end, 3):
        try:
            load_data = client.query_load(country_code, start=chunk_start, end=chunk_end)

            # Convert to DataFrame if needed
            if isinstance(load_data, list):
                load_data = pd.DataFrame(load_data)

            if not load_data.empty:
                load_data = load_data.iloc[:, -1]  # Keep only actual load column
                load_chunks.append(load_data)
        except Exception as e:
            print(f"Failed to fetch load data for {country_code} from {chunk_start} to {chunk_end}: {e}")

    if load_chunks:
        load_data = pd.concat(load_chunks)  # Merge chunks
        load_data = load_data.groupby(level=0).first()  # Remove duplicate timestamps
        all_loads[country_code] = load_data
    else:
        # Ensure missing country is represented with NaN
        all_loads[country_code] = pd.Series(dtype=float, name=country_code)

# Get price-specific timestamps
#price_index = pd.Index(sorted(set().union(*[df.index for df in all_prices.values()])), name="timestamp") if all_prices else pd.Index([], name="timestamp")

# Get load-specific timestamps
load_index = pd.Index(sorted(set().union(*[df.index for df in all_loads.values()])), name="timestamp") if all_loads else pd.Index([], name="timestamp")

# Align all price data with price-specific index
#df_prices = pd.DataFrame(index=price_index)
#for country, series in all_prices.items():
#    df_prices[country] = series.reindex(price_index)

# Align all load data with load-specific index
df_loads_HR = pd.DataFrame(index=load_index)
for country, series in all_loads.items():
    df_loads_HR[country] = series.reindex(load_index)

# Save to CSV
#df_prices.to_csv("electricity_prices_2022.csv", sep=";")
#print("Electricity prices saved to electricity_prices_2022.csv")

df_loads_HR.to_csv("../data/electricity_loads_HR_2022.csv", sep=";")
print("Electricity loads saved to electricity_loads_HR_2022.csv")



Fetching data for HR...
Electricity loads saved to electricity_loads_HR_2022.csv


In [13]:
df_loads_HR

,HR
timestamp,
2022-01-01 00:00:00+01:00,1783.0
2022-01-01 01:00:00+01:00,1676.0
2022-01-01 02:00:00+01:00,1579.0
2022-01-01 03:00:00+01:00,1503.0
2022-01-01 04:00:00+01:00,1462.0
...,...
2022-12-30 19:00:00+01:00,2244.0
2022-12-30 20:00:00+01:00,2197.0
2022-12-30 21:00:00+01:00,2164.0
